# Feature Engineering for EV Charging Sessions

The EDA notebook established what the raw file contains and how inconsistent it is. This
notebook turns those 1,320 raw rows into the single **model-ready table** that every
Phase-2 model reads from: `data/processed/sessions_clean.parquet`.

> **Feature engineering**
>
> Feature engineering is the step where raw columns are transformed into inputs a model
> can use: parsing a timestamp into an hour and a weekday, combining two columns into a
> ratio, or turning a category label into numbers. Good features encode domain knowledge
> that the model would otherwise have to discover on its own from limited data.

We build features in three independent groups — calendar parts, physical-consistency
ratios, and categorical encodings — show each transformation and its result, and check
each one with an `assert`. The same functions live in `evcharging.features` so the
production pipeline and this notebook stay identical.

## Contents

* Load and inspect the raw sessions
* Calendar features from the real timestamps
* Physical-consistency ratios (the anomaly-detection groundwork)
* Encoding the categorical columns
* The combined `build_features` pipeline
* Validation flags
* Missing-value policy, demonstrated
* Persist the processed dataset

## Load and inspect

`load_raw` reads the CSV, renames the headers to `snake_case`, parses the timestamps,
coerces the numeric columns, and adds `duration_hours` = `end_time − start_time`. It does
not drop rows or fill gaps.

In [1]:
import sys

sys.path.insert(0, "../src")

import numpy as np
import pandas as pd

from evcharging.data import load_raw

df = load_raw()

In [2]:
df

,user_id,vehicle_model,battery_capacity_kwh,station_id,location,start_time,end_time,energy_kwh,duration_hours_reported,charging_rate_kw,...,time_of_day_reported,day_of_week_reported,soc_start_pct,soc_end_pct,distance_km,temperature_c,vehicle_age_years,charger_type,user_type,duration_hours
0,User_1,BMW i3,108.463007,Station_391,Houston,2024-01-01 00:00:00,2024-01-01 00:39:00,60.712346,0.591363,36.389181,...,Evening,Tuesday,29.371576,86.119962,293.602111,27.947953,2.0,DC Fast Charger,Commuter,0.650000
1,User_2,Hyundai Kona,100.000000,Station_428,San Francisco,2024-01-01 01:00:00,2024-01-01 03:01:00,12.339275,3.133652,30.677735,...,Morning,Monday,10.115778,84.664344,112.112804,14.311026,3.0,Level 1,Casual Driver,2.016667
2,User_3,Chevy Bolt,75.000000,Station_181,San Francisco,2024-01-01 02:00:00,2024-01-01 04:48:00,19.128876,2.452653,27.513593,...,Morning,Thursday,6.854604,69.917615,71.799253,21.002002,2.0,Level 2,Commuter,2.800000
3,User_4,Hyundai Kona,50.000000,Station_327,Houston,2024-01-01 03:00:00,2024-01-01 06:42:00,79.457824,1.266431,32.882870,...,Evening,Saturday,83.120003,99.624328,199.577785,38.316313,1.0,Level 1,Long-Distance Traveler,3.700000
4,User_5,Hyundai Kona,50.000000,Station_108,Los Angeles,2024-01-01 04:00:00,2024-01-01 05:46:00,19.629104,2.019765,10.215712,...,Morning,Saturday,54.258950,63.743786,203.661847,-7.834199,1.0,Level 1,Long-Distance Traveler,1.766667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1315,User_1316,Nissan Leaf,100.000000,Station_57,New York,2024-02-24 19:00:00,2024-02-24 20:30:00,42.011654,1.426444,5.895475,...,Evening,Sunday,39.204102,83.915952,239.601075,1.919655,7.0,DC Fast Charger,Commuter,1.500000
1316,User_1317,BMW i3,100.000000,Station_40,New York,2024-02-24 20:00:00,2024-02-24 20:44:00,68.185853,3.238212,18.388012,...,Evening,Tuesday,31.456375,93.096461,164.376022,34.029775,4.0,Level 2,Casual Driver,0.733333
1317,User_1318,Nissan Leaf,100.000000,Station_374,New York,2024-02-24 21:00:00,2024-02-24 23:03:00,18.895102,3.267122,45.482066,...,Evening,Tuesday,71.903081,78.678879,226.519258,20.358761,5.0,DC Fast Charger,Commuter,2.050000
1318,User_1319,Chevy Bolt,85.000000,Station_336,San Francisco,2024-02-24 22:00:00,2024-02-24 23:20:00,13.756252,2.754527,38.148183,...,Afternoon,Sunday,76.187997,65.926573,291.494076,24.134598,5.0,Level 2,Commuter,1.333333


In [3]:
print(df.shape)
print(df.dtypes)

(1320, 21)
user_id                            object
vehicle_model                      object
battery_capacity_kwh              float64
station_id                         object
location                           object
start_time                 datetime64[ns]
end_time                   datetime64[ns]
energy_kwh                        float64
duration_hours_reported           float64
charging_rate_kw                  float64
cost_usd                          float64
time_of_day_reported               object
day_of_week_reported               object
soc_start_pct                     float64
soc_end_pct                       float64
distance_km                       float64
temperature_c                     float64
vehicle_age_years                 float64
charger_type                       object
user_type                          object
duration_hours                    float64
dtype: object


In [4]:
# The three columns with gaps, carried through unchanged
df[["energy_kwh", "charging_rate_kw", "distance_km"]].isna().sum()

energy_kwh          66
charging_rate_kw    66
distance_km         66
dtype: int64

1,320 rows, 21 columns (20 original + `duration_hours`). Timestamps are `datetime64`, the
measurements are floats, and the 66 + 66 + 66 missing values are still present. This is
the input to every step below.

## Calendar features from the real timestamps

The raw `time_of_day_reported` and `day_of_week_reported` columns do not agree with the
timestamps (shown in the EDA notebook), so we derive our own calendar features from
`start_time`, which is always exactly on the hour:

* `hour` — 0–23
* `weekday` — 0 (Monday) to 6 (Sunday)
* `day_name` — the weekday spelled out, for readable plots
* `month` — 1 or 2 (the data covers January–February 2024)
* `is_weekend` — 1 on Saturday/Sunday, else 0

`hour`, `weekday` and `is_weekend` are the calendar inputs to the demand-forecasting
model; the recommendation engine uses them to describe a proposed charging window.

In [5]:
from evcharging.features import add_time_parts

df = add_time_parts(df)
df[["start_time", "hour", "weekday", "day_name", "month", "is_weekend"]].head()

,start_time,hour,weekday,day_name,month,is_weekend
0,2024-01-01 00:00:00,0,0,Monday,1,0
1,2024-01-01 01:00:00,1,0,Monday,1,0
2,2024-01-01 02:00:00,2,0,Monday,1,0
3,2024-01-01 03:00:00,3,0,Monday,1,0
4,2024-01-01 04:00:00,4,0,Monday,1,0


In [6]:
# Ranges are valid and nothing is missing
assert df["hour"].between(0, 23).all()
assert df["weekday"].between(0, 6).all()
assert set(df["month"].unique()) <= {1, 2}
assert (df["is_weekend"] == (df["weekday"] >= 5).astype(int)).all()
assert df[["hour", "weekday", "day_name", "month", "is_weekend"]].notna().all().all()

In [7]:
df["is_weekend"].value_counts()

is_weekend
0    960
1    360
Name: count, dtype: int64

The five new columns are complete and in range. About two-sevenths of sessions fall on a
weekend, as expected from an even spread across the week — there is no weekday/weekend
behavioural split in this data, so `is_weekend` is kept mainly for the forecasting
calendar rather than as a strong predictor.

## Physical-consistency ratios

Two identities hold for any real charging session:

$$Energy \approx ChargingRate \times Duration \qquad
SOC_{increase} \approx \frac{Energy}{BatteryCapacity} \times 100$$

Here *Energy* is kWh delivered, *ChargingRate* is average power in kW, *Duration* is the
session length in hours, *SOC* is battery state of charge in percent, and
*BatteryCapacity* is the pack size in kWh.

`add_consistency_features` rearranges each identity into a ratio that equals **1.0 when
the two sides agree**, plus a few raw derived quantities:

| Column | Definition | Reads as |
| --- | --- | --- |
| `soc_delta_pct` | `soc_end_pct − soc_start_pct` | percentage points added |
| `soc_implied_energy_kwh` | `soc_delta_pct/100 × battery_capacity_kwh` | kWh the SOC swing implies |
| `implied_power_kw` | `energy_kwh / duration_hours` | average power the session implies |
| `energy_per_km` | `energy_kwh / distance_km` | consumption rate |
| `power_consistency_ratio` | `charging_rate_kw × duration_hours / energy_kwh` | 1.0 = rate/duration/energy agree |
| `soc_energy_consistency_ratio` | `soc_implied_energy_kwh / energy_kwh` | 1.0 = SOC swing matches energy |
| `duration_consistency_ratio` | `duration_hours_reported / duration_hours` | 1.0 = reported matches timestamps |

Division by zero or by a missing value produces `NaN`, never `inf`.

In [8]:
from evcharging.features import add_consistency_features

df = add_consistency_features(df)
ratio_cols = ["power_consistency_ratio", "soc_energy_consistency_ratio",
              "duration_consistency_ratio"]
df[["soc_delta_pct", "implied_power_kw", "energy_per_km", *ratio_cols]].describe()

,soc_delta_pct,implied_power_kw,energy_per_km,power_consistency_ratio,soc_energy_consistency_ratio,duration_consistency_ratio
count,1320.000000,1254.000000,1193.000000,1191.000000,1254.000000,1320.000000
mean,26.011578,26.023600,0.555622,3.417954,1.154455,1.347893
std,29.814729,23.482344,1.165946,34.179669,15.022934,1.122255
min,-84.360948,0.028312,0.000168,0.028089,-46.632695,0.032679
25%,4.808443,10.132382,0.144797,0.525579,0.054870,0.636403
50%,24.996490,19.386101,0.278421,1.200744,0.423853,1.000406
75%,47.721784,32.807490,0.537465,2.574519,0.909476,1.692200
max,127.463125,155.397580,22.120096,1160.139736,526.936294,8.918357


In [9]:
# soc_delta_pct is exactly end - start
assert np.allclose(df["soc_delta_pct"], df["soc_end_pct"] - df["soc_start_pct"])

# No infinities leaked into the ratio columns (NaN is allowed, inf is not)
assert not np.isinf(df[ratio_cols].to_numpy(dtype="float64", na_value=np.nan)).any()

In [10]:
# How far each consistency ratio sits from the ideal value of 1.0
(df[ratio_cols].median() - 1).abs().rename("median distance from 1.0")

power_consistency_ratio         0.200744
soc_energy_consistency_ratio    0.576147
duration_consistency_ratio      0.000406
Name: median distance from 1.0, dtype: float64

If the dataset were internally consistent, all three ratios would sit tightly on 1.0.
Instead every one has a median away from 1.0 and a very wide spread — `power_consistency_ratio`
in particular ranges over three orders of magnitude. These columns are the raw material
for the Isolation-Forest anomaly model in notebook `06`, which learns what a "normal"
combination of these ratios looks like and scores each session against it.

## Encoding the categorical columns

Models need numbers, not label strings. We use two schemes:

* **One-hot encoding** for the nominal fields `vehicle_model`, `location`, `user_type` —
  each value becomes its own 0/1 indicator column. We keep every category (no
  `drop_first`) so the columns remain self-describing for tree models and for the
  dashboard.
* **Ordinal encoding** for `charger_type`, which has a genuine physical speed order:
  `charger_type_code` is 0 for Level 1, 1 for Level 2, 2 for DC Fast Charger.

The original string columns are kept alongside the encodings.

In [11]:
from evcharging.features import add_encodings

df = add_encodings(df)
df.filter(regex="^(vehicle_model_|location_|user_type_|charger_type_code$)").head()

,vehicle_model_BMW i3,vehicle_model_Chevy Bolt,vehicle_model_Hyundai Kona,vehicle_model_Nissan Leaf,vehicle_model_Tesla Model 3,location_Chicago,location_Houston,location_Los Angeles,location_New York,location_San Francisco,user_type_Casual Driver,user_type_Commuter,user_type_Long-Distance Traveler,charger_type_code
0,1,0,0,0,0,0,1,0,0,0,0,1,0,2
1,0,0,1,0,0,0,0,0,0,1,1,0,0,0
2,0,1,0,0,0,0,0,0,0,1,0,1,0,1
3,0,0,1,0,0,0,1,0,0,0,0,0,1,0
4,0,0,1,0,0,0,0,1,0,0,0,0,1,0


In [12]:
# One indicator per category, and they sum to 1 across each group
for prefix, n in [("vehicle_model_", 5), ("location_", 5), ("user_type_", 3)]:
    cols = df.filter(regex=f"^{prefix}").columns
    assert len(cols) == n
    assert (df[cols].sum(axis=1) == 1).all()

# Ordinal code respects the speed order
codes = df.groupby("charger_type")["charger_type_code"].first()
assert codes["Level 1"] < codes["Level 2"] < codes["DC Fast Charger"]
codes

charger_type
DC Fast Charger    2
Level 1            0
Level 2            1
Name: charger_type_code, dtype: Int64

The encodings are complete and consistent: exactly one indicator is set per categorical
group, and the charger-type code increases with charging speed. Thirteen new numeric
columns are now available to the models.

## The combined `build_features` pipeline

`build_features` simply runs the three steps above in order. The production scripts and
the API call this one function; running it here on a fresh load must reproduce the frame
we assembled step by step.

In [13]:
from evcharging.features import build_features

features = build_features(load_raw())
print(features.shape)

(1320, 47)


In [14]:
# The step-by-step frame and the one-call frame are identical
pd.testing.assert_frame_equal(
    features.reset_index(drop=True), df.reset_index(drop=True)
)
print("build_features reproduces the manual steps exactly")

build_features reproduces the manual steps exactly


## Validation flags

The nine domain-rule flags from `evcharging.data.validate` are attached as boolean
columns, plus `flag_any` (their row-wise OR). They travel with the processed dataset so
that every downstream task — and the final honesty section of the README — can see, per
row, exactly which rules a session breaks.

In [15]:
from evcharging.data import add_validation_flags, build_validation_report
from evcharging.data.validate import FLAG_COLUMNS

features = add_validation_flags(features)
report = build_validation_report(features)
pd.DataFrame(report["rules"]).T

,description,count,pct
flag_energy_exceeds_capacity,Energy consumed exceeds the vehicle's battery ...,190,14.39
flag_soc_not_increasing,State of charge did not increase during the se...,268,20.3
flag_soc_out_of_range,A state-of-charge reading falls outside 0-100 %.,32,2.42
flag_duration_mismatch,Reported duration differs from (end - start) b...,951,72.05
flag_power_mismatch,|rate x duration - energy| / energy exceeds 50%.,783,59.32
flag_soc_energy_mismatch,|SOC-implied energy - measured energy| / energ...,871,65.98
flag_battery_capacity_out_of_range,Battery capacity outside 10.0-150.0 kWh.,13,0.98
flag_temperature_out_of_range,Temperature outside -30.0 to 60.0 C.,2,0.15
flag_missing_values,"At least one of energy_kwh, charging_rate_kw, ...",189,14.32


In [16]:
assert features[FLAG_COLUMNS].dtypes.eq(bool).all()
assert features["flag_any"].equals(features[FLAG_COLUMNS].any(axis=1))
print(f"{report['n_rows_flagged']} / {report['n_rows']} rows "
      f"({report['pct_rows_flagged']}%) break at least one rule")

1296 / 1320 rows (98.18%) break at least one rule


Almost every row — around 98% — breaks at least one rule, driven mostly by the duration
and the two energy-consistency mismatches; only about two dozen sessions are fully clean.
We still keep every row: dropping flagged sessions would leave nothing to model and would
remove the exact signal the anomaly detector is meant to learn.

## Missing-value policy, demonstrated

The processed file keeps missing values as `NaN`. Imputation is deferred to each model's
own preparation step, because the right choice depends on the target:

1. drop a row only when *that model's* target is missing, then
2. fill the remaining predictor gaps on the modelling subset — numeric to the median,
   categorical to `"unknown"` — with `impute_predictors`.

Here is that policy applied for the energy regressor, whose target is `energy_kwh`.

In [17]:
from evcharging.features import impute_predictors

target = "energy_kwh"
predictors_num = ["battery_capacity_kwh", "charging_rate_kw", "distance_km",
                  "temperature_c", "vehicle_age_years", "soc_start_pct", "soc_end_pct"]

# 1. drop rows where the target itself is missing
model_df = features.dropna(subset=[target]).copy()
print(f"rows: {len(features)} -> {len(model_df)} after dropping missing target")

# 2. impute predictor gaps on this subset only
before = model_df[predictors_num].isna().sum().sum()
model_df = impute_predictors(model_df, numeric_cols=predictors_num)
after = model_df[predictors_num].isna().sum().sum()
print(f"predictor NaNs: {before} -> {after}")

rows: 1320 -> 1254 after dropping missing target
predictor NaNs: 124 -> 0


In [18]:
# Target is complete, predictors are complete, medians came from this subset
assert model_df[target].notna().all()
assert model_df[predictors_num].notna().all().all()

Dropping on the target removes 66 rows; imputing predictors on the remaining 1,254 fills
the rest. Because the medians are computed on the modelling subset, not the full file, a
model never sees a value influenced by rows it would not have trained on. Notebooks `03`
onward repeat this two-step prep with their own target.

## Persist the processed dataset

Finally we write the two artifacts the rest of the project depends on. Both are
git-ignored and fully regenerated by `python scripts/prepare_data.py`, which runs exactly
the steps in this notebook.

In [19]:
from evcharging.config import CLEAN_PARQUET, VALIDATION_REPORT
from evcharging.data import write_validation_report

CLEAN_PARQUET.parent.mkdir(parents=True, exist_ok=True)
features.to_parquet(CLEAN_PARQUET, index=False)
write_validation_report(features, VALIDATION_REPORT)

print("wrote", CLEAN_PARQUET.name, "and", VALIDATION_REPORT.name)

wrote sessions_clean.parquet and validation_report.json


In [20]:
# Reload and confirm the round-trip preserved shape, row order and flags
reloaded = pd.read_parquet(CLEAN_PARQUET)
assert reloaded.shape == features.shape
assert reloaded["user_id"].tolist() == features["user_id"].tolist()
assert reloaded[FLAG_COLUMNS].equals(features[FLAG_COLUMNS])
reloaded.shape

(1320, 57)

## Summary / Key Takeaways

* The processed dataset is **1,320 rows × ~57 columns**: the original fields, five
  calendar features, seven physical-consistency quantities, thirteen categorical
  encodings, and ten validation-flag columns.
* Features are built in three pure, independently testable steps; `build_features` chains
  them and is what production code calls.
* `duration_hours` (from timestamps) replaces the unreliable reported duration
  everywhere.
* Missing values are **not** imputed in the processed file. Each model drops rows on its
  own target and calls `impute_predictors` on the remainder.
* The consistency ratios are deliberately centred so that 1.0 means "physically
  consistent"; their wide spread here is the input to the anomaly model.
* `data/processed/sessions_clean.parquet` and `validation_report.json` are now written
  and verified. Notebook `03_energy_prediction.ipynb` picks up from this file.